# 02 — Exploratory Data Analysis: Pollen Time Series

Exploratory analysis of daily pollen concentrations across 4 climatically contrasted French cities (2022–2024).

This notebook establishes the statistical properties of the data that will guide model selection in notebook 03.

**Key questions:**
1. What distributional family fits pollen concentrations? → modelling likelihood choice
2. Is there deterministic seasonality? → seasonal decomposition strategy
3. Are seasonal dynamics homogeneous across cities? → core research question
4. Which weather covariates correlate with pollen? → exogenous regressor selection
5. Are the series stationary (after differencing)? → ARIMA feasibility
6. What ACF/PACF signatures emerge? → ARIMA order hints

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, kruskal, shapiro
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Unified Style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "0.8",
    "legend.fontsize": 9,
})

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_PROC = os.path.join("..", "Data", "processed")
FIG_DIR = os.path.join("..", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# ── Configuration ────────────────────────────────────────────────────────────
CITIES = {
    "Paris":      {"climate": "Oceanic/Continental"},
    "Marseille":  {"climate": "Mediterranean"},
    "Strasbourg": {"climate": "Continental"},
    "Bordeaux":   {"climate": "Oceanic"},
}

CITY_COLORS = {
    "Paris": "#4C72B0", "Marseille": "#DD8452",
    "Strasbourg": "#55A868", "Bordeaux": "#C44E52",
}

POLLEN_SPECIES = [
    "birch_pollen", "alder_pollen", "grass_pollen",
    "olive_pollen", "mugwort_pollen", "ragweed_pollen",
]
SPECIES_LABELS = {s: s.replace("_pollen", "").capitalize() for s in POLLEN_SPECIES}
SPECIES_COLORS = {
    "grass_pollen": "#55A868", "birch_pollen": "#8C564B",
    "olive_pollen": "#E5AE38", "alder_pollen": "#64B5CD",
    "mugwort_pollen": "#8172B3", "ragweed_pollen": "#DA8BC3",
}

FOCUS_SPECIES = ["grass_pollen", "birch_pollen", "olive_pollen"]
FOCUS_COLORS = {s: SPECIES_COLORS[s] for s in FOCUS_SPECIES}

WEATHER_VARS = [
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "precipitation_sum", "windspeed_10m_max", "sunshine_duration",
    "relative_humidity_2m_mean",
    "surface_pressure_mean", "shortwave_radiation_sum", "winddirection_10m_dominant",
]

YEAR_COLORS = {2022: "#4C72B0", 2023: "#DD8452", 2024: "#55A868"}

print("Setup complete.")

Setup complete.


In [2]:
# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_csv(os.path.join(DATA_PROC, "pollen_weather_merged.csv"), parse_dates=["date"])
spin_annual = pd.read_csv(os.path.join(DATA_PROC, "spin_annual.csv"), parse_dates=["season_start", "season_end", "peak_date"])
spin_monthly = pd.read_csv(os.path.join(DATA_PROC, "spin_monthly.csv"))

print(f"Main dataset: {df.shape} — {df['date'].min().date()} → {df['date'].max().date()}")
print(f"SPIn annual:  {spin_annual.shape}")
print(f"SPIn monthly: {spin_monthly.shape}")
df.head()

Main dataset: (4384, 25) — 2022-01-01 → 2024-12-31
SPIn annual:  (72, 9)
SPIn monthly: (864, 5)


,date,birch_pollen,alder_pollen,grass_pollen,olive_pollen,mugwort_pollen,ragweed_pollen,city,temperature_2m_mean,temperature_2m_max,...,surface_pressure_mean,shortwave_radiation_sum,winddirection_10m_dominant,ozone,nitrogen_dioxide,pm2_5,climate,month,year,day_of_year
0,2022-01-01,0.0,0.0,0.0,0.0,0.0,0.0,Bordeaux,10.822917,18.50,...,1022.68805,6.75,148.07648,18.291666,16.616667,16.583334,Oceanic,1,2022,1
1,2022-01-02,0.0,0.0,0.0,0.0,0.0,0.0,Bordeaux,11.891666,15.90,...,1023.53143,5.61,224.04399,40.833332,12.287499,10.429167,Oceanic,1,2022,2
2,2022-01-03,0.0,0.0,0.0,0.0,0.0,0.0,Bordeaux,11.595834,15.95,...,1020.14410,6.05,180.34985,41.250000,12.041667,8.770833,Oceanic,1,2022,3
3,2022-01-04,0.0,0.0,0.0,0.0,0.0,0.0,Bordeaux,12.393750,15.85,...,1006.52985,3.28,245.08568,53.083332,8.537499,6.704166,Oceanic,1,2022,4
4,2022-01-05,0.0,0.0,0.0,0.0,0.0,0.0,Bordeaux,4.925000,8.00,...,1015.84300,7.13,327.46130,40.000000,16.900000,8.933333,Oceanic,1,2022,5


## 1 — Distributional Properties

Before choosing a modelling framework, we must understand the statistical distribution of pollen concentrations. The key moments are:

**Skewness** (asymmetry):

$$\gamma_1 = \frac{\mu_3}{\sigma^3} = \frac{\mathbb{E}[(X - \mu)^3]}{(\mathbb{E}[(X - \mu)^2])^{3/2}}$$

- $\gamma_1 = 0$: symmetric (e.g. Gaussian)
- $\gamma_1 \gg 0$: right-skewed — heavy right tail, most values concentrated near zero

**Excess kurtosis** (tail heaviness):

$$\gamma_2 = \frac{\mu_4}{\sigma^4} - 3 = \frac{\mathbb{E}[(X - \mu)^4]}{(\mathbb{E}[(X - \mu)^2])^2} - 3$$

- $\gamma_2 = 0$: mesokurtic (Gaussian)
- $\gamma_2 > 0$: leptokurtic — heavier tails, more extreme values than Gaussian

Pollen data is expected to be **strongly right-skewed** ($\gamma_1 \gg 0$) and **leptokurtic** ($\gamma_2 \gg 0$) because most days have zero or very low concentrations, with rare intense peaks during the active season. This has direct modelling implications: standard Gaussian models (OLS, ARIMA with normal errors) may perform poorly without a variance-stabilising transformation.

In [3]:
# ── 1.1 Summary statistics table ─────────────────────────────────────────────
summary_records = []
for city in CITIES:
    city_df = df[df["city"] == city]
    for species in POLLEN_SPECIES:
        vals = city_df[species].values
        summary_records.append({
            "City": city,
            "Species": SPECIES_LABELS[species],
            "Mean": np.mean(vals),
            "Std": np.std(vals),
            "Median": np.median(vals),
            "Max": np.max(vals),
            "Skewness": stats.skew(vals),
            "Kurtosis": stats.kurtosis(vals),  # excess kurtosis
            "% Zero": 100 * np.mean(vals == 0),
        })

summary_df = pd.DataFrame(summary_records)
summary_df = summary_df.round(2)

# Display pivot for readability
print("=== Distributional Summary (all cities × species) ===")
display(summary_df.style.format(precision=2).background_gradient(
    subset=["Skewness", "% Zero"], cmap="YlOrRd"
))

=== Distributional Summary (all cities × species) ===


,City,Species,Mean,Std,Median,Max,Skewness,Kurtosis,% Zero
0,Paris,Birch,3.24,14.89,0.00,145.48,6.35,44.44,80.11
1,Paris,Alder,0.16,1.01,0.00,17.50,12.04,172.59,80.84
2,Paris,Grass,3.91,8.65,0.00,55.58,2.88,8.50,53.28
3,Paris,Olive,0.02,0.15,0.00,2.62,12.67,171.04,95.71
4,Paris,Mugwort,3.14,7.64,0.00,63.64,3.44,13.85,61.13
5,Paris,Ragweed,0.15,0.65,0.00,9.62,7.15,67.59,84.31
6,Marseille,Birch,0.66,2.64,0.00,42.33,7.76,84.19,74.82
7,Marseille,Alder,0.13,0.50,0.00,7.91,6.93,70.61,72.81
8,Marseille,Grass,1.72,3.67,0.12,29.36,3.67,16.18,43.80
9,Marseille,Olive,0.21,1.04,0.00,13.18,8.05,76.48,80.47


In [4]:
# ── 1.2 Distribution histograms (focus species × cities) ─────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle("Pollen Concentration Distributions (log y-axis)", fontsize=15, fontweight="bold")

for row, species in enumerate(FOCUS_SPECIES):
    for col, city in enumerate(CITIES):
        ax = axes[row, col]
        vals = df[(df["city"] == city)][species].values
        nonzero = vals[vals > 0]

        ax.hist(nonzero, bins=50, color=FOCUS_COLORS[species], alpha=0.7, edgecolor="white")
        ax.set_yscale("log")
        ax.axvline(np.median(nonzero), color="black", ls="--", lw=1, label=f"median={np.median(nonzero):.1f}")

        # Inset stats
        ax.text(0.95, 0.95, f"γ₁={stats.skew(vals):.1f}\n{np.mean(vals==0)*100:.0f}% zeros",
                transform=ax.transAxes, ha="right", va="top", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

        if row == 0:
            ax.set_title(city, fontsize=12)
        if col == 0:
            ax.set_ylabel(SPECIES_LABELS[species], fontsize=11)
        ax.legend(fontsize=8, loc="upper right", framealpha=0.5) if col == 3 else None

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "02_eda_pollen_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2160x1440 with 12 Axes>

### 1.3 — Log-transform assessment

Given the strong right-skew, we apply the **variance-stabilising transformation**:

$$y_t = \log(1 + x_t)$$

**Why $\log(1 + x)$ instead of $\log(x)$?** — Because $x_t = 0$ is frequent (zero-inflation), and $\log(0)$ is undefined. The shift by 1 ensures the transform is defined on $[0, +\infty)$ and maps 0 → 0.

**Variance stabilisation intuition**: For a right-skewed variable where $\text{Var}[X]$ grows with $\mathbb{E}[X]$ (heteroscedasticity), taking $\log$ compresses the upper tail, making $\text{Var}[\log(1+X)]$ approximately constant. Formally, by the **delta method**, if $g(X) = \log(1+X)$:

$$\text{Var}[g(X)] \approx [g'(\mu)]^2 \cdot \text{Var}[X] = \frac{\sigma^2}{(1+\mu)^2}$$

So if $\sigma^2 \propto (1+\mu)^2$ (variance proportional to squared mean — common for count-like data), then $\text{Var}[\log(1+X)] \approx \text{const}$.

We assess the transform by comparing skewness before/after and checking QQ-plots against the normal distribution on **non-zero values only** (the zero mass is a separate structural component).

In [5]:
# ── 1.3 Log-transform: skewness comparison + QQ-plots ────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle("QQ-Plots: log(1 + x) Transformed Pollen (non-zero values)", fontsize=15, fontweight="bold")

skew_comparison = []

for row, species in enumerate(FOCUS_SPECIES):
    for col, city in enumerate(CITIES):
        ax = axes[row, col]
        vals = df[df["city"] == city][species].values
        log_vals = np.log1p(vals)
        nonzero_log = log_vals[vals > 0]

        # QQ-plot
        stats.probplot(nonzero_log, dist="norm", plot=ax)
        ax.get_lines()[0].set(markersize=2, alpha=0.5, color=FOCUS_COLORS[species])
        ax.get_lines()[1].set(color="black", linewidth=1)

        skew_comparison.append({
            "City": city, "Species": SPECIES_LABELS[species],
            "Skew (raw)": round(stats.skew(vals), 2),
            "Skew (log1p)": round(stats.skew(log_vals), 2),
            "Skew (log1p, >0)": round(stats.skew(nonzero_log), 2),
        })

        if row == 0:
            ax.set_title(city, fontsize=12)
        if col == 0:
            ax.set_ylabel(SPECIES_LABELS[species], fontsize=11)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "03_eda_log_transform_qq.png"), dpi=150, bbox_inches="tight")
plt.show()

# Skewness comparison table
skew_df = pd.DataFrame(skew_comparison)
print("=== Skewness: raw vs log(1+x) ===")
display(skew_df.style.background_gradient(subset=["Skew (raw)", "Skew (log1p)"], cmap="YlOrRd"))

<Figure size 2160x1440 with 12 Axes>

=== Skewness: raw vs log(1+x) ===


,City,Species,Skew (raw),Skew (log1p),"Skew (log1p, >0)"
0,Paris,Grass,2.880000,1.340000,0.310000
1,Marseille,Grass,3.670000,1.370000,0.720000
2,Strasbourg,Grass,2.870000,1.340000,0.300000
3,Bordeaux,Grass,3.230000,1.060000,0.410000
4,Paris,Birch,6.350000,3.230000,0.620000
5,Marseille,Birch,7.760000,3.090000,0.990000
6,Strasbourg,Birch,6.590000,2.840000,0.520000
7,Bordeaux,Birch,6.330000,3.570000,0.970000
8,Paris,Olive,12.670000,11.360000,1.810000
9,Marseille,Olive,8.050000,4.660000,1.650000


### 1.4 — Zero-inflation analysis

Pollen concentration data has a **two-part structure**:

$$X_t = \begin{cases} 0 & \text{with probability } \pi \quad \text{(off-season / no pollen)} \\ Y_t \sim f(\cdot) & \text{with probability } 1 - \pi \quad \text{(active season)} \end{cases}$$

where $\pi$ is the zero-inflation probability and $f(\cdot)$ is a continuous distribution for positive concentrations.

If $\pi$ is very high (e.g. > 70%), a **hurdle model** (separate binary + continuous components) may outperform a single continuous model. If $\pi$ is moderate and the $\log(1+x)$ transform handles the skew adequately, a standard time series model on the transformed data may suffice.

We examine:
1. The percentage of zero days per species/city
2. The distribution of the **non-zero portion** to characterize $f(\cdot)$

In [6]:
# ── 1.4 Zero-inflation analysis ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Zero-Inflation Analysis", fontsize=15, fontweight="bold")

# Panel 1: % zero days per species × city
zero_pct = []
for species in POLLEN_SPECIES:
    for city in CITIES:
        vals = df[df["city"] == city][species].values
        zero_pct.append({
            "Species": SPECIES_LABELS[species],
            "City": city,
            "% Zero": 100 * np.mean(vals == 0),
        })
zero_df = pd.DataFrame(zero_pct)
pivot = zero_df.pivot(index="Species", columns="City", values="% Zero")
pivot = pivot[list(CITIES.keys())]  # consistent order
pivot.plot(kind="bar", ax=ax1, color=[CITY_COLORS[c] for c in CITIES], edgecolor="white")
ax1.set_ylabel("% Zero Days")
ax1.set_title("Zero-inflation by species and city")
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right")
ax1.axhline(50, color="gray", ls="--", lw=0.8, alpha=0.5)
ax1.legend(title="City", fontsize=9)

# Panel 2: Violin of non-zero values (log scale) for focus species
violin_data = []
for species in FOCUS_SPECIES:
    for city in CITIES:
        vals = df[df["city"] == city][species].values
        nonzero = vals[vals > 0]
        for v in nonzero:
            violin_data.append({"Species": SPECIES_LABELS[species], "City": city, "log1p": np.log1p(v)})
violin_df = pd.DataFrame(violin_data)

sns.violinplot(data=violin_df, x="Species", y="log1p", hue="City",
               palette=CITY_COLORS, ax=ax2, inner="quartile", cut=0)
ax2.set_ylabel("log(1 + concentration)")
ax2.set_title("Non-zero distribution (log-scale)")
ax2.legend(title="City", fontsize=9, loc="upper right")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "04_eda_zero_inflation.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2160x720 with 2 Axes>

## 2 — Temporal Patterns & Seasonality

A time series $\{x_t\}_{t=1}^{n}$ can be decomposed into deterministic and stochastic components:

$$x_t = T_t + S_t + R_t \quad \text{(additive)}$$

$$x_t = T_t \cdot S_t \cdot R_t \quad \text{(multiplicative)}$$

where:
- $T_t$ = **trend** (long-term directional change)
- $S_t$ = **seasonal component** with period $s$ such that $S_t = S_{t+s}$ for all $t$
- $R_t$ = **residual** (stochastic, what ARIMA will model)

**Choice of model**: The additive model is appropriate when the seasonal amplitude is roughly constant over time. If the amplitude scales with the level (larger peaks in higher-concentration years), the multiplicative model is more appropriate. For pollen data, since we will work with $\log(1+x)$ (which already stabilises variance), the additive decomposition on the transformed series is the natural choice.

We use **STL** (Seasonal-Trend decomposition using LOESS), which fits local weighted polynomials:

$$\hat{f}(x_0) = \arg\min_{\beta_0, \beta_1} \sum_{i=1}^{n} K_h(x_i - x_0) \left(y_i - \beta_0 - \beta_1(x_i - x_0)\right)^2$$

where $K_h$ is a **tricube kernel** with bandwidth $h$:

$$K_h(u) = \begin{cases} \left(1 - |u/h|^3\right)^3 & \text{if } |u| \leq h \\ 0 & \text{otherwise} \end{cases}$$

STL iteratively extracts $T_t$ (by smoothing the de-seasonalised series) and $S_t$ (by smoothing the cycle-subseries), leaving $R_t = x_t - T_t - S_t$.

In [7]:
# ── 2.1 Full time series plots (focus species) ───────────────────────────────
for species in FOCUS_SPECIES:
    fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
    fig.suptitle(f"{SPECIES_LABELS[species]} Pollen — Daily Concentration (grains/m³)",
                 fontsize=14, fontweight="bold")

    for ax, city in zip(axes, CITIES):
        city_df = df[df["city"] == city].sort_values("date")
        ax.plot(city_df["date"], city_df[species], linewidth=0.6,
                color=CITY_COLORS[city], alpha=0.8)
        ax.fill_between(city_df["date"], city_df[species], alpha=0.15, color=CITY_COLORS[city])
        ax.set_ylabel("grains/m³", fontsize=9)
        ax.set_title(f"{city} ({CITIES[city]['climate']})", fontsize=11, loc="left")
        ax.grid(True, alpha=0.3)

    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.xticks(rotation=45)
    plt.tight_layout()
    fname = f"05_eda_timeseries_{SPECIES_LABELS[species].lower()}.png"
    fig.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()

<Figure size 1920x1440 with 4 Axes>

<Figure size 1920x1440 with 4 Axes>

<Figure size 1920x1440 with 4 Axes>

In [8]:
# ── 2.2 Seasonal subseries: monthly boxplots ─────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle("Monthly Boxplots of Pollen Concentration (Focus Species × City)",
             fontsize=15, fontweight="bold")

for row, species in enumerate(FOCUS_SPECIES):
    for col, city in enumerate(CITIES):
        ax = axes[row, col]
        city_df = df[df["city"] == city]
        sns.boxplot(data=city_df, x="month", y=species, ax=ax,
                    color=FOCUS_COLORS[species], fliersize=2, linewidth=0.8)
        ax.set_xlabel("")
        ax.set_ylabel("grains/m³" if col == 0 else "")
        if row == 0:
            ax.set_title(city, fontsize=12)
        if col == 0:
            ax.annotate(SPECIES_LABELS[species], xy=(-0.35, 0.5),
                        xycoords="axes fraction", fontsize=12, fontweight="bold",
                        rotation=90, va="center")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "08_eda_seasonal_boxplots.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2400x1680 with 12 Axes>

In [9]:
# ── 2.3 Year-over-year overlay (grass pollen) ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Grass Pollen — Year-over-Year Overlay by Day of Year", fontsize=14, fontweight="bold")
year_colors = {2022: "#636EFA", 2023: "#EF553B", 2024: "#00CC96"}

for ax, city in zip(axes.flat, CITIES):
    city_df = df[df["city"] == city].sort_values("date")
    for year in [2022, 2023, 2024]:
        yr_df = city_df[city_df["year"] == year]
        ax.plot(yr_df["day_of_year"], yr_df["grass_pollen"],
                linewidth=0.8, alpha=0.8, color=year_colors[year], label=str(year))
    ax.set_title(f"{city} ({CITIES[city]['climate']})", fontsize=11)
    ax.set_xlabel("Day of Year")
    ax.set_ylabel("grains/m³")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "09_eda_yoy_overlay_grass.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1920x1200 with 4 Axes>

In [10]:
# ── 2.4 STL Decomposition (grass pollen, all cities) ─────────────────────────
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
fig.suptitle("STL Decomposition — Grass Pollen log(1+x), period=365",
             fontsize=15, fontweight="bold")

component_names = ["Observed", "Trend", "Seasonal", "Residual"]

for col, city in enumerate(CITIES):
    city_df = df[df["city"] == city].sort_values("date").set_index("date")
    series = np.log1p(city_df["grass_pollen"])

    result = seasonal_decompose(series, model="additive", period=365,
                                extrapolate_trend="freq")

    for row, (component, name) in enumerate(zip(
        [result.observed, result.trend, result.seasonal, result.resid],
        component_names
    )):
        ax = axes[row, col]
        ax.plot(component.index, component.values, linewidth=0.5,
                color=CITY_COLORS[city])
        ax.grid(True, alpha=0.3)
        if col == 0:
            ax.set_ylabel(name, fontsize=11, fontweight="bold")
        if row == 0:
            ax.set_title(city, fontsize=12)
        ax.tick_params(axis="x", labelsize=8, rotation=45)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "10_eda_stl_decomposition_grass.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2400x1920 with 16 Axes>

In [11]:
# ── 2.5 Phenology heatmap (day-of-year × year) ───────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle("Grass Pollen Phenology Heatmap (Day of Year × Year)", fontsize=14, fontweight="bold")

for ax, city in zip(axes, CITIES):
    city_df = df[df["city"] == city].copy()
    pivot = city_df.pivot_table(index="year", columns="day_of_year",
                                 values="grass_pollen", aggfunc="mean")
    sns.heatmap(pivot, ax=ax, cmap="YlGn", cbar_kws={"label": "grains/m³", "shrink": 0.8},
                xticklabels=30, yticklabels=True)
    ax.set_title(city, fontsize=11)
    ax.set_xlabel("Day of Year")
    ax.set_ylabel("Year" if city == "Paris" else "")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "11_eda_phenology_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2400x480 with 8 Axes>

## 3 — Cross-City Comparison

> *"Are seasonal dynamics homogeneous across climatically contrasted regions?"*

### Statistical framework

Since pollen concentrations are non-Gaussian, we use the **Kruskal-Wallis H-test** — a non-parametric alternative to one-way ANOVA that compares the rank distributions of $k$ groups:

$$H = \frac{12}{N(N+1)} \sum_{i=1}^{k} \frac{R_i^2}{n_i} - 3(N+1)$$

where:
- $N$ = total number of observations across all groups
- $n_i$ = number of observations in group $i$
- $R_i$ = sum of ranks in group $i$

Under $H_0$ (all groups drawn from the same distribution): $H \sim \chi^2_{k-1}$

**Effect size** (eta-squared for Kruskal-Wallis):

$$\eta^2_H = \frac{H - k + 1}{N - k}$$

Interpretation: $\eta^2 < 0.01$ negligible, $0.01$–$0.06$ small, $0.06$–$0.14$ medium, $> 0.14$ large.

**Multiple testing correction**: Since we test 6 species simultaneously, we apply the **Bonferroni correction**:

$$\alpha_{\text{adj}} = \frac{\alpha}{m} = \frac{0.05}{6} \approx 0.0083$$

to control the family-wise error rate.

In [12]:
# ── 3.1 Superimposed seasonal profiles ────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 14))
fig.suptitle("Average Seasonal Profiles — 3-Year Mean ± Std by Day of Year",
             fontsize=15, fontweight="bold")

for ax, species in zip(axes, FOCUS_SPECIES):
    for city in CITIES:
        city_df = df[df["city"] == city]
        # Mean and std across years for each day_of_year
        grouped = city_df.groupby("day_of_year")[species]
        mean = grouped.mean()
        std = grouped.std()

        ax.plot(mean.index, mean.values, linewidth=1.5,
                color=CITY_COLORS[city], label=city)
        ax.fill_between(mean.index, (mean - std).clip(lower=0), mean + std,
                         alpha=0.1, color=CITY_COLORS[city])

    ax.set_title(SPECIES_LABELS[species], fontsize=12, loc="left")
    ax.set_ylabel("grains/m³")
    ax.legend(fontsize=10, loc="upper right")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Day of Year")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "12_eda_seasonal_profiles_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1920x1680 with 3 Axes>

In [13]:
# ── 3.2 Kruskal-Wallis test per species ───────────────────────────────────────
alpha = 0.05
m_tests = len(POLLEN_SPECIES)
alpha_adj = alpha / m_tests  # Bonferroni

kw_results = []
for species in POLLEN_SPECIES:
    groups = [df[df["city"] == city][species].values for city in CITIES]
    H_stat, p_value = kruskal(*groups)
    N = sum(len(g) for g in groups)
    k = len(groups)
    eta_sq = (H_stat - k + 1) / (N - k)

    kw_results.append({
        "Species": SPECIES_LABELS[species],
        "H-statistic": round(H_stat, 2),
        "p-value": f"{p_value:.2e}",
        "η² (effect size)": round(eta_sq, 4),
        "Effect": "Large" if eta_sq > 0.14 else "Medium" if eta_sq > 0.06 else "Small" if eta_sq > 0.01 else "Negligible",
        f"Significant (α={alpha_adj:.4f})": "Yes" if p_value < alpha_adj else "No",
    })

kw_df = pd.DataFrame(kw_results)
print("=== Kruskal-Wallis Test: Cross-City Homogeneity (Bonferroni-corrected) ===")
display(kw_df)

=== Kruskal-Wallis Test: Cross-City Homogeneity (Bonferroni-corrected) ===


,Species,H-statistic,p-value,η² (effect size),Effect,Significant (α=0.0083)
0,Birch,8.29,4.03e-02,0.0012,Negligible,No
1,Alder,21.52,8.22e-05,0.0042,Negligible,Yes
2,Grass,35.78,8.34e-08,0.0075,Negligible,Yes
3,Olive,158.70,3.50e-34,0.0355,Small,Yes
4,Mugwort,58.80,1.06e-12,0.0127,Small,Yes
5,Ragweed,15.79,1.25e-03,0.0029,Negligible,Yes


In [14]:
# ── 3.3 Season timing Gantt chart ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
fig.suptitle("Pollen Season Timing (5%–95% Cumulative Method)", fontsize=15, fontweight="bold")

focus_species_short = ["grass", "birch", "olive"]

for ax, sp in zip(axes, focus_species_short):
    sp_data = spin_annual[spin_annual["species"] == sp].copy()
    sp_data = sp_data.dropna(subset=["season_start", "season_end"])

    y_labels = []
    y_pos = 0
    for year in [2022, 2023, 2024]:
        for city in CITIES:
            row = sp_data[(sp_data["city"] == city) & (sp_data["year"] == year)]
            if not row.empty:
                row = row.iloc[0]
                start = pd.Timestamp(row["season_start"])
                end = pd.Timestamp(row["season_end"])
                duration = (end - start).days
                ax.barh(y_pos, duration, left=start.dayofyear, height=0.7,
                        color=CITY_COLORS[city], alpha=0.8, edgecolor="white")
                # Peak marker
                if pd.notna(row["peak_date"]):
                    peak_doy = pd.Timestamp(row["peak_date"]).dayofyear
                    ax.plot(peak_doy, y_pos, "k*", markersize=8)
            y_labels.append(f"{city} {year}")
            y_pos += 1

    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=8)
    ax.set_title(sp.capitalize(), fontsize=12, loc="left")
    ax.set_xlim(1, 365)
    ax.grid(True, axis="x", alpha=0.3)
    # Month markers
    month_starts = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    ax.set_xticks(month_starts)
    ax.set_xticklabels(month_names, fontsize=9)

axes[-1].set_xlabel("Month")
# Legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=CITY_COLORS[c], label=c) for c in CITIES]
legend_handles.append(plt.Line2D([], [], marker="*", color="black", linestyle="None", markersize=8, label="Peak"))
axes[0].legend(handles=legend_handles, loc="upper right", fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "13_eda_season_timing_gantt.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1920x1440 with 3 Axes>

In [15]:
# ── 3.4 SPIn comparison bar chart ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Annual SPIn by City and Year (Focus Species)", fontsize=15, fontweight="bold")

for ax, sp in zip(axes, focus_species_short):
    sp_data = spin_annual[spin_annual["species"] == sp]
    pivot = sp_data.pivot_table(index="city", columns="year", values="spin")
    pivot = pivot.reindex(list(CITIES.keys()))
    pivot.plot(kind="bar", ax=ax, color=["#636EFA", "#EF553B", "#00CC96"],
               edgecolor="white")
    ax.set_title(sp.capitalize(), fontsize=12)
    ax.set_ylabel("SPIn (grains·day/m³)")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.legend(title="Year", fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "14_eda_spin_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2160x720 with 3 Axes>

In [16]:
# ── 3.5 Peak concentration & timing analysis ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Peak Concentration vs. Peak Timing (Day of Year)", fontsize=15, fontweight="bold")

for ax, sp in zip(axes, focus_species_short):
    sp_data = spin_annual[spin_annual["species"] == sp].dropna(subset=["peak_date"])
    sp_data = sp_data.copy()
    sp_data["peak_doy"] = sp_data["peak_date"].dt.dayofyear

    for city in CITIES:
        city_data = sp_data[sp_data["city"] == city]
        ax.scatter(city_data["peak_doy"], city_data["peak_concentration"],
                   color=CITY_COLORS[city], s=80, label=city, edgecolor="white", zorder=3)
        # Annotate with year
        for _, row in city_data.iterrows():
            ax.annotate(str(row["year"]), (row["peak_doy"], row["peak_concentration"]),
                        fontsize=7, ha="center", va="bottom")

    ax.set_title(sp.capitalize(), fontsize=12)
    ax.set_xlabel("Peak Day of Year")
    ax.set_ylabel("Peak Concentration (grains/m³)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "15_eda_peak_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2160x720 with 3 Axes>

## 4 — SPIn Deep Dive

Recall the **Seasonal Pollen Integral**:

$$\text{SPIn} = \sum_{d=d_{\text{start}}}^{d_{\text{end}}} C_d$$

with season bounds at cumulative 5%/95% of the annual total (computed in notebook 01).

### Inter-annual variability

The **coefficient of variation** (CV) quantifies relative variability:

$$\text{CV} = \frac{\sigma}{\mu}$$

where $\sigma$ and $\mu$ are the standard deviation and mean of annual SPIn values across the 3 years. 
- $\text{CV} < 0.3$: relatively stable seasons — a single seasonal template may work well for forecasting
- $\text{CV} > 1$: highly variable — year-specific predictors (weather) will be critical

In [17]:
# ── 4.1 Monthly SPIn heatmap ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle("Monthly SPIn Distribution (City-Year × Month)", fontsize=15, fontweight="bold")

for ax, sp in zip(axes, focus_species_short):
    sp_data = spin_monthly[spin_monthly["species"] == sp].copy()
    sp_data["label"] = sp_data["city"] + " " + sp_data["year"].astype(str)
    pivot = sp_data.pivot_table(index="label", columns="month", values="spin_monthly", fill_value=0)
    # Sort by city then year
    order = [f"{c} {y}" for c in CITIES for y in [2022, 2023, 2024]]
    pivot = pivot.reindex([o for o in order if o in pivot.index])

    sns.heatmap(pivot, ax=ax, cmap="YlOrRd", annot=False, fmt=".0f",
                cbar_kws={"label": "SPIn", "shrink": 0.7},
                xticklabels=["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
    ax.set_title(sp.capitalize(), fontsize=12)
    ax.set_ylabel("")
    ax.set_xlabel("Month")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "16_eda_spin_monthly_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2400x960 with 6 Axes>

In [18]:
# ── 4.2 Season duration vs SPIn scatter ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
fig.suptitle("Season Duration vs. SPIn", fontsize=14, fontweight="bold")

markers = {"grass": "o", "birch": "s", "olive": "D"}
for sp in focus_species_short:
    sp_data = spin_annual[(spin_annual["species"] == sp) & (spin_annual["spin"] > 0)]
    for city in CITIES:
        cd = sp_data[sp_data["city"] == city]
        ax.scatter(cd["season_duration_days"], cd["spin"],
                   color=CITY_COLORS[city], marker=markers[sp], s=80,
                   edgecolor="white", alpha=0.8)

# Compute overall Spearman
valid = spin_annual[spin_annual["spin"] > 0]
rho, p = spearmanr(valid["season_duration_days"], valid["spin"])
ax.set_xlabel("Season Duration (days)")
ax.set_ylabel("SPIn (grains·day/m³)")
ax.set_title(f"Spearman ρ = {rho:.3f} (p = {p:.2e})", fontsize=11)
ax.grid(True, alpha=0.3)

# Custom legend
from matplotlib.lines import Line2D
city_handles = [Line2D([], [], marker="o", color=CITY_COLORS[c], linestyle="None", markersize=8, label=c) for c in CITIES]
sp_handles = [Line2D([], [], marker=markers[sp], color="gray", linestyle="None", markersize=8, label=sp.capitalize()) for sp in focus_species_short]
ax.legend(handles=city_handles + sp_handles, fontsize=9, loc="upper left")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "17_eda_spin_duration_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1200x840 with 1 Axes>

In [19]:
# ── 4.3 Inter-annual variability (CV) ─────────────────────────────────────────
cv_records = []
for sp in [s.replace("_pollen", "") for s in POLLEN_SPECIES]:
    sp_data = spin_annual[spin_annual["species"] == sp]
    for city in CITIES:
        cd = sp_data[sp_data["city"] == city]
        mu = cd["spin"].mean()
        sigma = cd["spin"].std()
        cv = sigma / mu if mu > 0 else np.nan
        cv_records.append({"Species": sp.capitalize(), "City": city, "CV": round(cv, 3)})

cv_df = pd.DataFrame(cv_records)
cv_pivot = cv_df.pivot(index="Species", columns="City", values="CV")
cv_pivot = cv_pivot[list(CITIES.keys())]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cv_pivot, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax,
            cbar_kws={"label": "CV (σ/μ)"}, linewidths=0.5)
ax.set_title("Inter-Annual Variability of SPIn (Coefficient of Variation)", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "18_eda_spin_interannual_variability.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1200x600 with 2 Axes>

## 5 — Weather-Pollen Correlations

Identifying which meteorological variables drive pollen concentrations is essential for building exogenous-regressor models (SARIMAX). We examine both **contemporaneous** and **lagged** correlations.

### Cross-correlation at lag $k$

$$r_{xy}(k) = \frac{\sum_{t=1}^{n-k} (x_t - \bar{x})(y_{t+k} - \bar{y})}{\sqrt{\sum_{t=1}^{n-k} (x_t - \bar{x})^2 \cdot \sum_{t=1}^{n-k} (y_{t+k} - \bar{y})^2}}$$

This measures the linear association between weather variable $x$ at time $t$ and pollen concentration $y$ at time $t+k$. A significant correlation at $k > 0$ means weather leads pollen by $k$ days — valuable for forecasting.

### Why Spearman over Pearson?

We use **Spearman's rank correlation** $\rho_s$ rather than Pearson's $r$ because:
1. Pollen data is non-Gaussian (right-skewed, zero-inflated)
2. Spearman captures **monotonic** relationships (not just linear)
3. It is **robust to outliers** (operates on ranks, not values)

$$\rho_s = 1 - \frac{6 \sum_{i=1}^{n} d_i^2}{n(n^2 - 1)}$$

where $d_i = \text{rank}(x_i) - \text{rank}(y_i)$.

### Season-only filtering

To avoid the correlation being dominated by the structural zero-inflation (both pollen and temperature are low in winter, high in summer — creating spurious correlation), we **restrict to pollen-season days** (defined as days within the 5%–95% cumulative window for the given species).

In [20]:
# ── Helper: get season mask from SPIn annual data ────────────────────────────
def get_season_mask(city_df, city, species_short, spin_df):
    """Return boolean mask for days within the pollen season (any year)."""
    mask = np.zeros(len(city_df), dtype=bool)
    for year in city_df["year"].unique():
        row = spin_df[(spin_df["city"] == city) & (spin_df["species"] == species_short) & (spin_df["year"] == year)]
        if row.empty or pd.isna(row.iloc[0]["season_start"]):
            continue
        start = pd.Timestamp(row.iloc[0]["season_start"])
        end = pd.Timestamp(row.iloc[0]["season_end"])
        year_mask = (city_df["date"] >= start) & (city_df["date"] <= end)
        mask |= year_mask.values
    return mask

In [21]:
# ── 5.1 Spearman correlation matrices (season-only) ──────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("Spearman Correlation: Pollen × Weather (Season-Only Days)", fontsize=15, fontweight="bold")

weather_labels = [v.replace("temperature_2m_", "T ").replace("_sum", "").replace("_10m_max", " max")
                  .replace("_duration", "").replace("relative_humidity_2m_mean", "RH mean") for v in WEATHER_VARS]

for ax, city in zip(axes.flat, CITIES):
    city_df = df[df["city"] == city].sort_values("date").reset_index(drop=True)
    corr_matrix = np.zeros((len(POLLEN_SPECIES), len(WEATHER_VARS)))

    for i, species in enumerate(POLLEN_SPECIES):
        sp_short = species.replace("_pollen", "")
        mask = get_season_mask(city_df, city, sp_short, spin_annual)
        if mask.sum() < 10:
            corr_matrix[i, :] = np.nan
            continue
        season_data = city_df[mask]
        for j, wvar in enumerate(WEATHER_VARS):
            rho, _ = spearmanr(season_data[species], season_data[wvar])
            corr_matrix[i, j] = rho

    sns.heatmap(corr_matrix, ax=ax, cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6,
                annot=True, fmt=".2f", xticklabels=weather_labels,
                yticklabels=[SPECIES_LABELS[s] for s in POLLEN_SPECIES],
                cbar_kws={"shrink": 0.7})
    ax.set_title(f"{city} ({CITIES[city]['climate']})", fontsize=12)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "19_eda_weather_pollen_correlations.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2160x1680 with 8 Axes>

### 5.1b — Multicollinearity Check (VIF)

Before injecting weather covariates into ARIMAX models (NB03), we must verify that the regressors are not **collinear**. A high Variance Inflation Factor ($\text{VIF}_j > 5$) signals that variable $j$ is well-predicted by the others, inflating coefficient standard errors and making the model unstable.

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ is the $R^2$ from regressing $x_j$ on all other regressors.

**Decision rule:** VIF > 10 → drop; VIF 5–10 → caution; VIF < 5 → acceptable.

In [22]:
# ── 5.1b VIF — Multicollinearity of weather regressors ───────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_records = []
for city in CITIES:
    city_df = df[df["city"] == city][WEATHER_VARS].dropna()
    for i, var in enumerate(WEATHER_VARS):
        vif_val = variance_inflation_factor(city_df.values, i)
        vif_records.append({"City": city, "Variable": var, "VIF": round(vif_val, 2)})

vif_df = pd.DataFrame(vif_records)
vif_pivot = vif_df.pivot(index="Variable", columns="City", values="VIF")
vif_pivot = vif_pivot[list(CITIES.keys())]

# Short labels for display
short_labels = {
    "temperature_2m_mean": "T mean", "temperature_2m_max": "T max",
    "temperature_2m_min": "T min", "precipitation_sum": "Precip",
    "windspeed_10m_max": "Wind max", "sunshine_duration": "Sunshine",
    "relative_humidity_2m_mean": "RH mean",
}
vif_pivot.index = [short_labels.get(v, v) for v in vif_pivot.index]

# Heatmap
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(vif_pivot, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, vmin=1, vmax=30,
            cbar_kws={"label": "VIF"})
ax.set_title("Variance Inflation Factor — Weather Regressors")
ax.set_ylabel("")

# Mark threshold
for i in range(vif_pivot.shape[0]):
    for j in range(vif_pivot.shape[1]):
        val = vif_pivot.iloc[i, j]
        if val > 10:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor="red", lw=2))

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "19b_eda_vif_weather.png"), bbox_inches="tight")
plt.show()

# Summary
print("=== VIF Summary ===")
print("Variables with VIF > 10 (high multicollinearity):")
high_vif = vif_df[vif_df["VIF"] > 10]
if len(high_vif) > 0:
    display(high_vif.sort_values("VIF", ascending=False))
    print("\n→ T mean, T max, T min are highly collinear (as expected).")
    print("  For ARIMAX: use T mean + sunshine (VIF < 5) to avoid collinearity.")
else:
    print("  None — all variables below threshold.")

print("\n→ Recommended ARIMAX regressors: temperature_2m_mean + sunshine_duration")

<Figure size 1200x600 with 2 Axes>

=== VIF Summary ===
Variables with VIF > 10 (high multicollinearity):


,City,Variable,VIF
10,Marseille,temperature_2m_mean,1412.08
30,Bordeaux,temperature_2m_mean,1176.87
0,Paris,temperature_2m_mean,936.97
20,Strasbourg,temperature_2m_mean,909.83
11,Marseille,temperature_2m_max,648.04
31,Bordeaux,temperature_2m_max,463.44
1,Paris,temperature_2m_max,445.19
21,Strasbourg,temperature_2m_max,407.06
12,Marseille,temperature_2m_min,293.98
7,Paris,surface_pressure_mean,266.06



→ T mean, T max, T min are highly collinear (as expected).
  For ARIMAX: use T mean + sunshine (VIF < 5) to avoid collinearity.

→ Recommended ARIMAX regressors: temperature_2m_mean + sunshine_duration


In [23]:
# ── 5.2 Lagged cross-correlation (grass vs key weather vars) ──────────────────
key_weather = ["temperature_2m_mean", "precipitation_sum", "sunshine_duration"]
key_labels = ["Temperature", "Precipitation", "Sunshine"]
max_lag = 14

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Lagged Cross-Correlation: Grass Pollen vs. Weather (lags 0–14 days)",
             fontsize=14, fontweight="bold")

for ax, city in zip(axes.flat, CITIES):
    city_df = df[df["city"] == city].sort_values("date").reset_index(drop=True)
    mask = get_season_mask(city_df, city, "grass", spin_annual)
    season = city_df[mask].copy()

    for wvar, wlabel in zip(key_weather, key_labels):
        lag_corrs = []
        for lag in range(max_lag + 1):
            if lag == 0:
                rho, _ = spearmanr(season["grass_pollen"], season[wvar])
            else:
                rho, _ = spearmanr(season["grass_pollen"].iloc[lag:],
                                   season[wvar].iloc[:-lag])
            lag_corrs.append(rho)
        ax.plot(range(max_lag + 1), lag_corrs, marker="o", markersize=4, label=wlabel)

    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_title(f"{city}", fontsize=11)
    ax.set_xlabel("Lag (days)")
    ax.set_ylabel("Spearman ρ")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "20_eda_lagged_crosscorrelation_grass.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1920x1200 with 4 Axes>

### 5.4 — Growing Degree Days (GDD)

Phenological models predict season onset based on **accumulated thermal units**:

$$\text{GDD}_t = \sum_{i=1}^{t} \max(T_i - T_{\text{base}}, 0), \quad T_{\text{base}} = 5°\text{C}$$

The rationale: plants require a minimum amount of cumulative warmth to break dormancy and begin flowering. GDD is reset to 0 at January 1 each year. We compare the GDD value at season onset (from SPIn data) across cities and years — if GDD at onset is consistent, it suggests thermal accumulation is a reliable predictor.

In [24]:
# ── 5.4 GDD vs season onset ───────────────────────────────────────────────────
T_BASE = 5.0

# Compute GDD for each city-year
gdd_at_onset = []
for city in CITIES:
    city_df = df[df["city"] == city].sort_values("date")
    for year in [2022, 2023, 2024]:
        yr_df = city_df[city_df["year"] == year].copy()
        yr_df["gdd_daily"] = (yr_df["temperature_2m_mean"] - T_BASE).clip(lower=0)
        yr_df["gdd_cumsum"] = yr_df["gdd_daily"].cumsum()

        for sp in focus_species_short:
            row = spin_annual[(spin_annual["city"] == city) & (spin_annual["species"] == sp) & (spin_annual["year"] == year)]
            if row.empty or pd.isna(row.iloc[0]["season_start"]):
                continue
            onset = pd.Timestamp(row.iloc[0]["season_start"])
            onset_row = yr_df[yr_df["date"] == onset]
            if not onset_row.empty:
                gdd_at_onset.append({
                    "City": city, "Species": sp.capitalize(), "Year": year,
                    "Onset DOY": onset.dayofyear,
                    "GDD at Onset": round(float(onset_row["gdd_cumsum"].iloc[0]), 1),
                })

gdd_df = pd.DataFrame(gdd_at_onset)

fig, ax = plt.subplots(figsize=(10, 7))
for sp in ["Grass", "Birch", "Olive"]:
    sp_data = gdd_df[gdd_df["Species"] == sp]
    for city in CITIES:
        cd = sp_data[sp_data["City"] == city]
        ax.scatter(cd["Onset DOY"], cd["GDD at Onset"],
                   color=CITY_COLORS[city], marker=markers[sp.lower()], s=100,
                   edgecolor="white", alpha=0.8)

ax.set_xlabel("Season Onset (Day of Year)")
ax.set_ylabel(f"Cumulative GDD at Onset (T_base = {T_BASE}°C)")
ax.set_title("Growing Degree Days at Season Onset", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)

# Legend
city_h = [Line2D([], [], marker="o", color=CITY_COLORS[c], ls="None", ms=8, label=c) for c in CITIES]
sp_h = [Line2D([], [], marker=markers[s], color="gray", ls="None", ms=8, label=s.capitalize()) for s in focus_species_short]
ax.legend(handles=city_h + sp_h, fontsize=9, loc="upper left")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "22_eda_gdd_season_onset.png"), dpi=150, bbox_inches="tight")
plt.show()

print("=== GDD at Season Onset ===")
display(gdd_df.pivot_table(index=["Species", "City"], columns="Year", values="GDD at Onset"))

<Figure size 1200x840 with 1 Axes>

=== GDD at Season Onset ===


Year                 2022   2023   2024
Species City                           
Birch   Bordeaux    244.6  229.7  319.4
        Marseille   303.1  267.2  357.0
        Paris       180.6  213.4  248.9
        Strasbourg  121.1  159.1  177.6
Grass   Bordeaux    524.1  456.1  585.7
        Marseille   633.6  585.0  711.9
        Paris       570.6  564.9  616.5
        Strasbourg  506.3  503.7  522.1
Olive   Bordeaux    708.7  526.0  491.1
        Marseille   772.0  627.6  734.1
        Paris       553.4  410.4  335.5
        Strasbourg  475.4  270.9  280.8

## 6 — Stationarity Tests

ARIMA-family models require **weak (second-order) stationarity**:
1. $\mathbb{E}[X_t] = \mu$ — constant mean
2. $\text{Cov}(X_t, X_{t+h}) = \gamma(h)$ — autocovariance depends only on lag $h$, not on $t$

### Augmented Dickey-Fuller (ADF) test

Tests for a **unit root** via the regression:

$$\Delta x_t = \alpha + \beta t + \gamma x_{t-1} + \sum_{i=1}^{p} \delta_i \Delta x_{t-i} + \varepsilon_t$$

- $H_0: \gamma = 0$ (unit root exists → non-stationary)
- $H_1: \gamma < 0$ (stationary)

Reject $H_0$ at $\alpha = 0.05$ → series is stationary.

### KPSS test (Kwiatkowski-Phillips-Schmidt-Shin)

Tests the **opposite hypothesis**:

$$x_t = \xi t + r_t + \varepsilon_t, \quad r_t = r_{t-1} + u_t$$

- $H_0: \sigma^2_u = 0$ (series is trend-stationary)
- $H_1: \sigma^2_u > 0$ (unit root → non-stationary)

Fail to reject $H_0$ → series is stationary.

### Combined decision matrix

| ADF result | KPSS result | Conclusion |
|:---:|:---:|:---|
| Reject $H_0$ | Fail to reject $H_0$ | **Stationary** |
| Fail to reject | Reject $H_0$ | **Non-stationary** |
| Reject | Reject | **Trend-stationary** (detrend needed) |
| Fail to reject | Fail to reject | **Inconclusive** |

### Differencing operators

If the raw series is non-stationary, we apply:
- **First difference**: $\nabla x_t = x_t - x_{t-1}$ (removes linear trend)
- **Seasonal difference**: $\nabla_s x_t = x_t - x_{t-s}$ (removes annual cycle, $s = 365$)
- **Combined**: $\nabla \nabla_s x_t = (x_t - x_{t-1}) - (x_{t-s} - x_{t-s-1})$

In [25]:
# ── 6.1 ADF + KPSS on raw and log(1+x) series ────────────────────────────────
def stationarity_tests(series, name=""):
    """Run ADF and KPSS, return summary dict."""
    series = series.dropna()
    # ADF
    adf_stat, adf_p, adf_lags, _, _, _ = adfuller(series, autolag="AIC")
    adf_reject = adf_p < 0.05
    # KPSS
    kpss_stat, kpss_p, kpss_lags, _ = kpss(series, regression="c", nlags="auto")
    kpss_reject = kpss_p < 0.05

    # Decision
    if adf_reject and not kpss_reject:
        conclusion = "Stationary"
    elif not adf_reject and kpss_reject:
        conclusion = "Non-stationary"
    elif adf_reject and kpss_reject:
        conclusion = "Trend-stationary"
    else:
        conclusion = "Inconclusive"

    return {
        "Series": name,
        "ADF stat": round(adf_stat, 3), "ADF p": round(adf_p, 4), "ADF reject H₀": adf_reject,
        "KPSS stat": round(kpss_stat, 3), "KPSS p": round(kpss_p, 4), "KPSS reject H₀": kpss_reject,
        "Conclusion": conclusion,
    }

# Run on focus species × cities — raw and log(1+x)
stat_results = []
for species in FOCUS_SPECIES:
    sp_label = SPECIES_LABELS[species]
    for city in CITIES:
        city_df = df[df["city"] == city].sort_values("date")
        raw = city_df[species]
        log = np.log1p(city_df[species])

        stat_results.append(stationarity_tests(raw, f"{sp_label} | {city} | raw"))
        stat_results.append(stationarity_tests(log, f"{sp_label} | {city} | log(1+x)"))

stat_df = pd.DataFrame(stat_results)
print("=== Stationarity Tests: Raw vs. Log(1+x) ===")
display(stat_df.style.applymap(
    lambda v: "background-color: #c6efce" if v == "Stationary" else
              "background-color: #ffc7ce" if v == "Non-stationary" else "",
    subset=["Conclusion"]
))

=== Stationarity Tests: Raw vs. Log(1+x) ===


,Series,ADF stat,ADF p,ADF reject H₀,KPSS stat,KPSS p,KPSS reject H₀,Conclusion
0,Grass | Paris | raw,-3.400000,0.011000,True,0.100000,0.100000,False,Stationary
1,Grass | Paris | log(1+x),-3.205000,0.019700,True,0.126000,0.100000,False,Stationary
2,Grass | Marseille | raw,-3.411000,0.010600,True,0.140000,0.100000,False,Stationary
3,Grass | Marseille | log(1+x),-2.229000,0.195900,False,0.143000,0.100000,False,Inconclusive
4,Grass | Strasbourg | raw,-4.600000,0.000100,True,0.095000,0.100000,False,Stationary
5,Grass | Strasbourg | log(1+x),-4.060000,0.001100,True,0.119000,0.100000,False,Stationary
6,Grass | Bordeaux | raw,-3.204000,0.019800,True,0.191000,0.100000,False,Stationary
7,Grass | Bordeaux | log(1+x),-1.905000,0.329900,False,0.177000,0.100000,False,Inconclusive
8,Birch | Paris | raw,-4.789000,0.000100,True,0.287000,0.100000,False,Stationary
9,Birch | Paris | log(1+x),-4.606000,0.000100,True,0.223000,0.100000,False,Stationary


In [26]:
# ── 6.2 ADF + KPSS on differenced series ──────────────────────────────────────
diff_results = []
for species in FOCUS_SPECIES:
    sp_label = SPECIES_LABELS[species]
    for city in CITIES:
        city_df = df[df["city"] == city].sort_values("date")
        log_series = np.log1p(city_df[species].values)

        # First difference (d=1)
        d1 = pd.Series(np.diff(log_series, n=1))
        diff_results.append(stationarity_tests(d1, f"{sp_label} | {city} | Δ log(1+x)"))

        # Seasonal difference (D=1, s=365)
        if len(log_series) > 365:
            ds = pd.Series(log_series[365:] - log_series[:-365])
            diff_results.append(stationarity_tests(ds, f"{sp_label} | {city} | Δ₃₆₅ log(1+x)"))

        # Combined (d=1, D=1)
        if len(log_series) > 366:
            d1s = pd.Series(np.diff(log_series[365:] - log_series[:-365], n=1))
            diff_results.append(stationarity_tests(d1s, f"{sp_label} | {city} | ΔΔ₃₆₅ log(1+x)"))

diff_df = pd.DataFrame(diff_results)
print("=== Stationarity Tests: Differenced Series ===")
display(diff_df.style.applymap(
    lambda v: "background-color: #c6efce" if v == "Stationary" else
              "background-color: #ffc7ce" if v == "Non-stationary" else "",
    subset=["Conclusion"]
))

=== Stationarity Tests: Differenced Series ===


,Series,ADF stat,ADF p,ADF reject H₀,KPSS stat,KPSS p,KPSS reject H₀,Conclusion
0,Grass | Paris | Δ log(1+x),-5.014000,0.000000,True,0.068000,0.100000,False,Stationary
1,Grass | Paris | Δ₃₆₅ log(1+x),-7.791000,0.000000,True,0.328000,0.100000,False,Stationary
2,Grass | Paris | ΔΔ₃₆₅ log(1+x),-10.048000,0.000000,True,0.020000,0.100000,False,Stationary
3,Grass | Marseille | Δ log(1+x),-8.138000,0.000000,True,0.062000,0.100000,False,Stationary
4,Grass | Marseille | Δ₃₆₅ log(1+x),-5.544000,0.000000,True,0.602000,0.022500,True,Trend-stationary
5,Grass | Marseille | ΔΔ₃₆₅ log(1+x),-8.393000,0.000000,True,0.054000,0.100000,False,Stationary
6,Grass | Strasbourg | Δ log(1+x),-3.903000,0.002000,True,0.085000,0.100000,False,Stationary
7,Grass | Strasbourg | Δ₃₆₅ log(1+x),-4.244000,0.000600,True,0.348000,0.099700,False,Stationary
8,Grass | Strasbourg | ΔΔ₃₆₅ log(1+x),-10.984000,0.000000,True,0.016000,0.100000,False,Stationary
9,Grass | Bordeaux | Δ log(1+x),-9.625000,0.000000,True,0.070000,0.100000,False,Stationary


In [27]:
# ── 6.3 Rolling statistics visual check ───────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle("Grass Pollen log(1+x) — Rolling Mean & Std (30-day window)", fontsize=14, fontweight="bold")

for ax, city in zip(axes, CITIES):
    city_df = df[df["city"] == city].sort_values("date").set_index("date")
    series = np.log1p(city_df["grass_pollen"])
    rolling_mean = series.rolling(window=30, center=True).mean()
    rolling_std = series.rolling(window=30, center=True).std()

    ax.plot(series.index, series.values, linewidth=0.4, alpha=0.5, color="gray", label="log(1+x)")
    ax.plot(rolling_mean.index, rolling_mean.values, linewidth=1.5, color=CITY_COLORS[city], label="Rolling mean (30d)")
    ax.fill_between(rolling_std.index,
                     rolling_mean - rolling_std, rolling_mean + rolling_std,
                     alpha=0.2, color=CITY_COLORS[city], label="±1 Rolling std")
    ax.set_ylabel("log(1+x)")
    ax.set_title(city, fontsize=11, loc="left")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "24_eda_rolling_statistics_grass.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1920x1680 with 4 Axes>

## 7 — Autocorrelation Analysis

The **autocorrelation function** (ACF) and **partial autocorrelation function** (PACF) are the primary diagnostic tools for identifying ARIMA orders.

### ACF

$$\hat{\rho}(k) = \frac{\hat{\gamma}(k)}{\hat{\gamma}(0)}, \quad \hat{\gamma}(k) = \frac{1}{n} \sum_{t=1}^{n-k} (x_t - \bar{x})(x_{t+k} - \bar{x})$$

### PACF

$\hat{\phi}_{kk}$ is the last coefficient of an AR($k$) model fitted to the data. It measures the correlation between $x_t$ and $x_{t+k}$ **after removing the linear dependence** on the intermediate values $x_{t+1}, \ldots, x_{t+k-1}$.

### ARIMA order identification

| ACF pattern | PACF pattern | Suggested model |
|:---:|:---:|:---|
| Cuts off at lag $q$ | Decays gradually | MA($q$) |
| Decays gradually | Cuts off at lag $p$ | AR($p$) |
| Both decay | Both decay | ARMA($p, q$) — use AIC/BIC |
| Spike at lag $s$ | Spike at lag $s$ | Seasonal component at period $s$ |

### Ljung-Box test

Tests whether autocorrelations up to lag $m$ are collectively zero:

$$Q(m) = n(n+2) \sum_{k=1}^{m} \frac{\hat{\rho}_k^2}{n - k} \sim \chi^2_m$$

Under $H_0$: data are independently distributed (white noise). Reject → significant autocorrelation remains.

In [28]:
# ── 7.1 ACF/PACF on raw log(1+x) grass pollen ────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(18, 16))
fig.suptitle("ACF & PACF — Grass Pollen log(1+x) (Raw)", fontsize=15, fontweight="bold")

for row, city in enumerate(CITIES):
    city_df = df[df["city"] == city].sort_values("date")
    series = np.log1p(city_df["grass_pollen"].values)

    ax_acf = axes[row, 0]
    ax_pacf = axes[row, 1]

    plot_acf(series, lags=400, ax=ax_acf, alpha=0.05, color=CITY_COLORS[city],
             vlines_kwargs={"colors": CITY_COLORS[city]})
    ax_acf.set_title(f"{city} — ACF", fontsize=11)

    plot_pacf(series, lags=50, ax=ax_pacf, alpha=0.05, color=CITY_COLORS[city],
              vlines_kwargs={"colors": CITY_COLORS[city]}, method="ywm")
    ax_pacf.set_title(f"{city} — PACF (lags 0–50)", fontsize=11)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "25_eda_acf_pacf_raw_grass.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2160x1920 with 8 Axes>

In [29]:
# ── 7.2 ACF/PACF on first-differenced log(1+x) grass pollen ──────────────────
fig, axes = plt.subplots(4, 2, figsize=(18, 16))
fig.suptitle("ACF & PACF — Grass Pollen Δlog(1+x) (First Differenced)", fontsize=15, fontweight="bold")

for row, city in enumerate(CITIES):
    city_df = df[df["city"] == city].sort_values("date")
    series = np.diff(np.log1p(city_df["grass_pollen"].values))

    ax_acf = axes[row, 0]
    ax_pacf = axes[row, 1]

    plot_acf(series, lags=400, ax=ax_acf, alpha=0.05, color=CITY_COLORS[city],
             vlines_kwargs={"colors": CITY_COLORS[city]})
    ax_acf.set_title(f"{city} — ACF (Δ)", fontsize=11)

    plot_pacf(series, lags=50, ax=ax_pacf, alpha=0.05, color=CITY_COLORS[city],
              vlines_kwargs={"colors": CITY_COLORS[city]}, method="ywm")
    ax_pacf.set_title(f"{city} — PACF (Δ, lags 0–50)", fontsize=11)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "26_eda_acf_pacf_diff_grass.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2160x1920 with 8 Axes>

In [30]:
# ── 7.3 ACF/PACF for birch and olive (compact) ───────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle("ACF — Birch & Olive Pollen log(1+x) (Raw)", fontsize=14, fontweight="bold")

for col, city in enumerate(CITIES):
    city_df = df[df["city"] == city].sort_values("date")

    for row, species in enumerate(["birch_pollen", "olive_pollen"]):
        ax = axes[row, col]
        series = np.log1p(city_df[species].values)
        plot_acf(series, lags=400, ax=ax, alpha=0.05, color=FOCUS_COLORS[species],
                 vlines_kwargs={"colors": FOCUS_COLORS[species]})
        if row == 0:
            ax.set_title(city, fontsize=11)
        if col == 0:
            ax.set_ylabel(SPECIES_LABELS[species])

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "27_eda_acf_birch_olive.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 2400x960 with 8 Axes>

In [31]:
# ── 7.4 Ljung-Box test ────────────────────────────────────────────────────────
lb_results = []
for species in FOCUS_SPECIES:
    sp_label = SPECIES_LABELS[species]
    for city in CITIES:
        city_df = df[df["city"] == city].sort_values("date")
        series = np.diff(np.log1p(city_df[species].values))  # first-differenced

        for m in [10, 20, 30]:
            lb = acorr_ljungbox(series, lags=[m], return_df=True)
            lb_results.append({
                "Species": sp_label, "City": city, "Lag m": m,
                "Q-stat": round(float(lb["lb_stat"].iloc[0]), 2),
                "p-value": f"{float(lb['lb_pvalue'].iloc[0]):.2e}",
                "Autocorrelation?": "Yes" if float(lb["lb_pvalue"].iloc[0]) < 0.05 else "No",
            })

lb_df = pd.DataFrame(lb_results)
print("=== Ljung-Box Test on Δlog(1+x) ===")
display(lb_df.pivot_table(index=["Species", "City"], columns="Lag m",
                           values="Q-stat", aggfunc="first"))

=== Ljung-Box Test on Δlog(1+x) ===


Lag m                   10      20      30
Species City                              
Birch   Bordeaux    126.43  171.86  251.59
        Marseille    60.25   92.03  141.58
        Paris       115.54  213.46  265.68
        Strasbourg  123.02  178.83  200.31
Grass   Bordeaux     77.91  115.48  134.41
        Marseille   111.82  181.81  240.18
        Paris        92.55  116.12  162.04
        Strasbourg  134.01  172.02  195.28
Olive   Bordeaux    309.08  323.16  326.06
        Marseille   105.09  137.22  159.32
        Paris       206.60  219.46  231.98
        Strasbourg  253.02  291.20  294.33

## 8 — Summary & Modelling Recommendations

### 8.1 — Key Findings

| Property | Finding | Modelling Implication |
|:---|:---|:---|
| **Distribution** | Right-skewed ($\gamma_1$ = 2.9–19.2), zero-inflated (42–96%), leptokurtic | Use $\log(1+x)$ transform for variance stabilisation |
| **Seasonality** | Strong annual cycle ($s = 365$), consistent across years | SARIMA with seasonal component |
| **Stationarity** | Most series already I(0) (ADF rejects $H_0$); grass in Marseille/Bordeaux borderline | $d = 1$ sufficient as safety, $D = 1$ optional |
| **Cross-city** | Kruskal-Wallis significant for all species **except birch** (Bonferroni) | Model each city separately; birch could potentially be pooled |
| **Strongest city effect** | Olive ($\eta^2 = 0.036$) — Mediterranean specificity confirmed | Olive models will differ most across cities |
| **Weather drivers** | Temperature + sunshine strongest; precipitation weaker | Include temp + sunshine as SARIMAX exogenous regressors |
| **GDD** | Cumulative temperature at onset varies by species but relatively stable within species | Potential phenological predictor for season onset timing |
| **Autocorrelation** | Ljung-Box highly significant at all lags after $\Delta$ — strong serial dependence | ARIMA AR/MA terms essential; seasonal ACF at ~365 |

### 8.2 — Recommended Modelling Strategy (Notebook 03)

Progressive benchmarking from simple → complex:

1. **Seasonal Naive** — $\hat{x}_{t+h} = x_{t+h-365}$ (repeat last year) — baseline
2. **SARIMA** — on $\log(1+x)$ transformed series, orders selected via `auto_arima`
3. **SARIMAX** — add temperature_2m_mean + sunshine_duration as exogenous regressors
4. **VAR** — multivariate model if cross-species correlations warrant it
5. **City-specific models** — confirmed by Kruskal-Wallis; no pooling

**Train/test split**: 2022–2023 for training, 2024 for out-of-sample forecasting.  
**Metrics**: RMSE, MAE, MAPE on the original (back-transformed) scale.